# Лабораторная работа №3 «Методы приближения функций»
Выполнил: Макаров Станислав Алексеевич, ИВТб-2301<br>
Вариант 14

In [ ]:
import numpy as np

from IPython.display import display, Markdown, HTML

def md(s):
    display(Markdown(s))

def tb(headers, rows):
    header_html = ''.join(f'<th>{header}</th>' for header in headers)
    body_html = ''.join(
        '<tr>' + ''.join(f'<td>{cell}</td>' for cell in row) + '</tr>'
        for row in rows
    )
    display(HTML(
        f'''
        <style>
          .table th,
          .table td {{
            border: 1px solid #000;
            min-width: 4rem;
            padding: 0.35rem 0.8rem;
            text-align: center;
          }}
        </style>
        <table class="table">
          <thead><tr>{header_html}</tr></thead>
          <tbody>{body_html}</tbody>
        </table>
        '''
    ))

def format_value(value):
    return f"{value:.5f}"

## Задание 1. Интерполяция по формуле Лагранжа

По таблице с неравностоящими значениями аргумента выполнить интерполяцию, используя формулу Лагранжа. Точность $\varepsilon=0.000001$:

In [2]:
X = np.array([0.41, 0.46, 0.52, 0.60, 0.65, 0.72])
Y = np.array([2.57418, 2.32513, 2.09336, 1.86203, 1.74926, 1.62098])
x = 0.478

md(f"$x={x}$")
tb([f"x", f"y"], [[x, y] for x, y in zip(X, Y)])

$x=0.478$

x,y
0.41,2.57418
0.46,2.32513
0.52,2.09336
0.6,1.86203
0.65,1.74926
0.72,1.62098


Напишем функцию, реализиующая формаулу Лагранжа $$L_n(x) = П_{n+1}(x)\sum_{i=0}^{n}\frac{y_i}{D_i}$$

In [3]:
def calculate_lagrange(x, X, Y, tb_func):
    n = len(X)
    X = np.array(X)
    Y = np.array(Y)

    diagonal_product = 1.0
    row_products = np.empty(n, dtype=float)

    headers = [f"x_{i}" for i in range(n)] + ['D_i * 10^6', 'y_i / D_i * 10^-6']
    table_rows = []

    for i in range(n):
        row_values = []
        current_row_prod = 1.0

        for j in range(n):
            if i == j:
                value = x - X[j]
                diagonal_product *= value
            else:
                value = X[i] - X[j]

            current_row_prod *= value
            row_values.append(f"{value:.5f}")

        row_products[i] = current_row_prod

        d_i = current_row_prod
        row_values.extend([f"{(d_i * 1e6):.5f}", f"{Y[i] / (d_i * -1e6):.5f}"])
        table_rows.append(row_values)

    md("Преобразованная таблица формулы Лагранжа")
    tb_func(headers, table_rows)

    return diagonal_product * np.sum(Y / row_products)

In [4]:
y = calculate_lagrange(x, X, Y, tb)

md(f"Значение $y$ при $x={x}$")
tb(["x", "y"], [[f"{x}", f"{y:.5f}"]])

Преобразованная таблица формулы Лагранжа

x_0,x_1,x_2,x_3,x_4,x_5,D_i * 10^6,y_i / D_i * 10^-6
0.06800,-0.05000,-0.11000,-0.19000,-0.24000,-0.31000,-5.28686,0.48690
0.05000,0.01800,-0.06000,-0.14000,-0.19000,-0.26000,0.37346,-6.22585
0.11000,0.06000,-0.04200,-0.08000,-0.13000,-0.20000,0.57658,-3.63067
0.19000,0.14000,0.08000,-0.12200,-0.05000,-0.12000,-1.55770,1.19537
0.24000,0.19000,0.13000,0.05000,-0.17200,-0.07000,3.56866,-0.49017
0.31000,0.26000,0.20000,0.12000,0.07000,-0.24200,-32.76874,0.04947


Значение $y$ при $x=0.478$

x,y
0.478,2.24899


## Задание 2. Интерполяция по формулам Ньютона

По таблице с равностоящими значениями аргумента вычислить значения функции для заданных значений аргументов, используя пеарвцю и вторую интерполяционные формулы Ньютона. Точность $\varepsilon=0.000001$:

In [50]:
X = np.array([0.180, 0.185, 0.190, 0.195, 0.200, 0.205, 0.210, 0.215, 0.220, 0.225, 0.230, 0.235])
Y = np.array([5.61543, 5.46698, 5.32634, 5.19304, 5.06649, 4.94619, 4.83170, 4.72261, 4.61855, 4.51912, 4.424222, 4.33337])
x = np.array([0.1827, 0.2292, 0.1776, 0.2405])

md("Исходные данные")
tb([f"x", f"y"], [[x, y] for x, y in zip(X, Y)])

md("Значения $x$")
tb([], [[f"{val:.5f}" for val in x]])

Исходные данные

x,y
0.18,5.61543
0.185,5.46698
0.19,5.32634
0.195,5.19304
0.2,5.06649
0.205,4.94619
0.21,4.8317
0.215,4.72261
0.22,4.61855
0.225,4.51912


Значения $x$

0.18270,0.22920,0.17760,0.24050


Напишем функцию для подсчета таблицы конечных разностей

In [ ]:
def difference_table(Y):
    table = [np.array(Y)]
    while table[-1].size > 1:
        table.append(np.diff(table[-1]))
    return table

Выведем полученную таблицу

In [53]:
def display_difference_table(X, Y):
    table = difference_table(Y)
  
    superscripts = {1: '', 2: '<sup>2</sup>', 3: '<sup>3</sup>', 4: '<sup>4</sup>', 5: '<sup>5</sup>', 6: '<sup>6</sup>', 7: '<sup>7</sup>', 8: '<sup>8</sup>', 9: '<sup>9</sup>'}
    headers = ['k', 'x<sub>k</sub>', 'y<sub>k</sub>'] + [
        'Δy<sub>k</sub>' if order == 1 else f"Δ{superscripts.get(order, f'<sup>{order}</sup>')}y<sub>k</sub>"
        for order in range(1, len(table))
    ]

    rows = []
    for k in range(X.size):
        row = [
            str(k), 
            f"{X[k]:.3f}", 
            f"{table[0][k]:.5f}"
        ]

        for order in range(1, len(table)):
            if k < table[order].size:
                row.append(f"{table[order][k]:.5f}")
            else:
                row.append("")

        rows.append(row)

    md("Таблица конечных разностей")
    tb(headers, rows)

display_difference_table(X, Y)

Таблица конечных разностей

k,xk,yk,Δyk,Δ2yk,Δ3yk,Δ4yk,Δ5yk,Δ6yk,Δ7yk,Δ8yk,Δ9yk,Δ10yk,Δ11yk
0,0.180,5.61543,-0.14845,0.00781,-0.00047,-0.00012,0.00021,-0.00024,0.00024,-0.00020,0.00004,0.00072,-0.00399
1,0.185,5.46698,-0.14064,0.00734,-0.00059,0.00009,-0.00003,-0.00000,0.00004,-0.00016,0.00076,-0.00327,
2,0.190,5.32634,-0.13330,0.00675,-0.00050,0.00006,-0.00003,0.00004,-0.00012,0.00060,-0.00251,,
3,0.195,5.19304,-0.12655,0.00625,-0.00044,0.00003,0.00001,-0.00008,0.00048,-0.00191,,,
4,0.200,5.06649,-0.12030,0.00581,-0.00041,0.00004,-0.00007,0.00040,-0.00142,,,,
5,0.205,4.94619,-0.11449,0.00540,-0.00037,-0.00003,0.00033,-0.00102,,,,,
6,0.210,4.83170,-0.10909,0.00503,-0.00040,0.00030,-0.00069,,,,,,
7,0.215,4.72261,-0.10406,0.00463,-0.00010,-0.00039,,,,,,,
8,0.220,4.61855,-0.09943,0.00453,-0.00049,,,,,,,,
9,0.225,4.51912,-0.09490,0.00405,,,,,,,,,


Напишем функцию для расчета по первой формуле Ньютона:
$$P_n(x)=y_0+q\Delta y_0+\frac{q(q-1)}{2!}\Delta^2 y_0+\dots+\frac{q(q-1)\cdots(q-n+1)}{n!}\Delta^n y_0$$

In [55]:
def newton_forward(x, X, Y):
    h = X[1] - X[0]
    table = difference_table(Y)
    p = p = (x - X[0]) / h
    result = table[0][0]
    factor = 1.0
    for order in range(1, len(Y)):
        factor *= (p - (order - 1)) / order
        result += factor * table[order][0]
    return result

И напишем для второй формулы Ньютона
$$P_n(x)=y_n+q\Delta y_{n-1}+\frac{q(q+1)}{2!}\Delta^2 y_{n-2}+\dots+\frac{q(q+1)\cdots(q+n-1)}{n!}\Delta^n y_0$$

In [56]:
def newton_backward(x, X, Y):
    h = X[1] - X[0]
    table = difference_table(Y)
    p = (x - X[-1]) / h
    result = table[0][-1]
    factor = 1.0
    for order in range(1, len(Y)):
        factor *= (p + (order - 1)) / order
        result += factor * table[order][-1]
    return result

Сделаем и отобразаим результаты интерполяции по формулам Ньютона 

In [57]:
def display_newton():
    newton_rows = []
    for x_i in x:
        y_forward = newton_forward(x_i, X, Y)
        y_backward = newton_backward(x_i, X, Y)
        newton_rows.append([
            f"{x_i:.4f}",
            f"{y_forward:.5f}",
            f"{y_backward:.5f}"
        ])

    md("Результаты интерполяции по формулам Ньютона")
    tb(
        ['x', 'Формула 1', 'Формула 2'],
        newton_rows
    )

display_newton()

Результаты интерполяции по формулам Ньютона

x,Формула 1,Формула 2
0.1827,5.53425,5.53425
0.2292,4.43909,4.43909
0.1776,5.69011,5.69011
0.2405,4.21756,4.21756


## Задание 3. Метод наименьших квадратов

По заданным экспериментальным точкам выбрать вид эмпирической звисимости и выполнить среднеквадратичное приближение функции, применив метод наименьших квадратов для оценки выбранной зависимости.

In [26]:
X = np.array([5.0, 5.2, 5.4, 5.6, 5.8, 6.0, 6.2, 6.4, 6.6, 6.8])
Y = np.array([20.2, 22.1, 24.1, 26.2, 28.3, 30.6, 33.0, 35.4, 37.9, 40.5])

In [30]:
def least_squares_linear(X, Y):
    """Метод наименьших квадратов"""
    n = X.size
    sum_x = X.sum()
    sum_y = Y.sum()
    sum_xx = np.square(X).sum()
    sum_xy = np.multiply(X, Y).sum()
    a = (n * sum_xy - sum_x * sum_y) / (n * sum_xx - sum_x ** 2)
    b = (sum_y - a * sum_x) / n
    return a, b

def least_squares_exponential(X, Y):
    """Показательная зависимость: y = a * b^x"""
    ln_y = np.log(Y)
    b, ln_a = least_squares_linear(X, ln_y)
    return np.exp(ln_a), np.exp(b)

def least_squares_inverse_linear(X, Y):
    """Дробно-линейная зависимость: y = 1 / (ax + b)"""
    transformed_y = 1.0 / Y
    a, b = least_squares_linear(X, transformed_y)
    return a, b

def least_squares_logarithmic(X, Y):
    """Логарифмическая зависимость: y = a ln(x) + b"""
    transformed_x = np.log(X)
    a, b = least_squares_linear(transformed_x, Y)
    return a, b

def least_squares_power(X, Y):
    """Степенная зависимость: y = a * x^b"""
    transformed_x = np.log(X)
    transformed_y = np.log(Y)
    b, ln_a = least_squares_linear(transformed_x, transformed_y)
    return np.exp(ln_a), b

def least_squares_hyperbolic(X, Y):
    """Гиперболическая зависимость: y = a + b/x"""
    transformed_x = 1.0 / X
    a, b = least_squares_linear(transformed_x, Y)
    return a, b

def least_squares_fractional_rational(x_values, y_values):
    """Подбирает параметры дробно-рациональной зависимости y = x / (ax + b)."""
    x_values = np.asarray(x_values, dtype=float)
    y_values = np.asarray(y_values, dtype=float)
    transformed_y = x_values / y_values
    a, b = least_squares_linear(x_values, transformed_y)
    return a, b

def sum_squared_errors(actual_values, predicted_values):
    actual_values = np.asarray(actual_values, dtype=float)
    predicted_values = np.asarray(predicted_values, dtype=float)
    return np.square(actual_values - predicted_values).sum()

In [31]:
models = [
    {
        "name": "Линейная",
        "fit": least_squares_linear,
        "pred": lambda x, a, b: a * x + b,
        "eq": lambda a, b: f"y = {format_value(a)}x {'+' if b>=0 else '-'} {format_value(abs(b))}"
    },
    {
        "name": "Показательная",
        "fit": least_squares_exponential,
        "pred": lambda x, a, b: a * (b ** x),
        "eq": lambda a, b: f"y = {format_value(a)} * {format_value(b)}^x"
    },
    {
        "name": "Дробно-линейная",
        "fit": least_squares_inverse_linear,
        "pred": lambda x, a, b: 1.0 / (a * x + b),
        "eq": lambda a, b: f"y = 1 / ({format_value(a)}x {'+' if b>=0 else '-'} {format_value(abs(b))})"
    },
    {
        "name": "Логарифмическая",
        "fit": least_squares_logarithmic,
        "pred": lambda x, a, b: a * np.log(x) + b,
        "eq": lambda a, b: f"y = {format_value(a)}ln(x) {'+' if b>=0 else '-'} {format_value(abs(b))}"
    },
    {
        "name": "Степенная",
        "fit": least_squares_power,
        "pred": lambda x, a, b: a * (x ** b),
        "eq": lambda a, b: f"y = {format_value(a)}x^{format_value(b)}"
    },
    {
        "name": "Гиперболическая",
        "fit": least_squares_hyperbolic,
        "pred": lambda x, a, b: a / x + b,
        "eq": lambda a, b: f"y = {format_value(b)} {'+' if a>=0 else '-'} {format_value(abs(a))} / x"
    },
    {
        "name": "Дробно-рациональная",
        "fit": least_squares_fractional_rational,
        "pred": lambda x, a, b: x / (a * x + b),
        "eq": lambda a, b: f"y = x / ({format_value(a)}x {'+' if b>=0 else '-'} {format_value(abs(b))})"
    }
]

results = []
n_points = len(X)

for model in models:
    a, b = model["fit"](X, Y)
    y_pred = model["pred"](X, a, b)

    sse = sum_squared_errors(Y, y_pred)
    sigma = float(np.sqrt(sse / n_points))

    results.append({
        "name": model["name"],
        "eq": model["eq"](a, b),
        "err": sigma
    })

tb(
    ['Модель', 'Уравнение', 'ε (σ)'],
    [[res["name"], res["eq"], format_value(res["err"])] for res in results],
)

Модель,Уравнение,ε (σ)
Линейная,y = 11.28788x - 36.76848,0.31741
Показательная,y = 2.99054 * 1.47069^x,0.32962
Дробно-линейная,y = 1 / (-0.01357x + 0.11528),1.19562
Логарифмическая,y = 65.95958ln(x) - 86.92979,0.59663
Степенная,y = 0.52971x^2.26344,0.04336
Гиперболическая,y = 95.16579 - 381.79951 / x,0.87508
Дробно-рациональная,y = x / (-0.04386x + 0.46203),0.44036
